# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the "A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa" using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant JSON-LD URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset name: {metadata.name}")
print(f"Description: {metadata.description}")

## 2. Data Overview
Explore the structure of the dataset: available record sets and field identifiers. All references are made using their `@id` fields as per the Croissant standard.

In [ ]:
# List all record sets and their fields by @id
record_sets = []
print("Available record sets and their field @id's:")
for record_set in metadata.record_sets:
    print(f"- Record Set @id: {record_set.id}  |  name: {record_set.name}")
    field_ids = [field.id for field in record_set.fields]
    print("  Field @id's in this Record Set:")
    for f in record_set.fields:
        print(f"    - {f.id} (name: {f.name}, data_type: {f.data_type})")
    record_sets.append(record_set.id)

if not record_sets:
    print("No top-level record sets found! The dataset may be using an alternative description or nesting. Inspecting via mlcroissant iteration:")
    print("Attempting ds.records() without specifying record_set...")
    # Try to get at least one record as an example
    try:
        for rec in dataset.records():
            print("Sample record keys:", list(rec.keys()))
            break
    except Exception as e:
        print("No records found or dataset not structured with explicit record sets.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the `@id` of the chosen record set and fields.

In [ ]:
# List of record set @id's to extract (from above; manual or code).
if record_sets:
    record_set_list = record_sets
else:
    # If record_sets wasn't populated, try to infer main table (common for tabular survey datasets)
    # mlcroissant typically allows records() to work for the default/main record set
    record_set_list = [None]

dataframes = {}

for record_set_id in record_set_list:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id if record_set_id else 'default'] = df
    print(f"Loaded {len(df)} records for Record Set: {record_set_id if record_set_id else 'default'}")
    print("Fields (columns):", df.columns.tolist())
    # Show head for the first (main) record set
    display(df.head())
    break  # Only show for the first for brevity

# For remaining steps, pick the main data table
main_record_set_id = record_set_list[0] if record_set_list else 'default'

## 4. Exploratory Data Analysis (EDA)
We'll filter, normalize, and explore relationships in the main survey data. Here we reference fields by their `@id` if possible. 

Let's inspect the available numeric and group fields for demonstration.

In [ ]:
df = dataframes[main_record_set_id]

# Try to pick a numeric field for EDA. If field metadata is available, prefer a mental health score.
numeric_candidates = [col for col in df.columns if df[col].dtype in ['int64', 'float64']]
if not numeric_candidates:
    # Try to pick known possible field names
    numeric_candidates = [col for col in df.columns if any(x in col.lower() for x in ['gad7', 'phq9', 'psq', 'score', 'age', 'total'])]

if numeric_candidates:
    numeric_field = numeric_candidates[0]
else:
    raise ValueError("No numeric fields detected in data.")

print(f"Numeric field chosen for EDA: {numeric_field}")

# Example filtering
threshold = 10
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold} (N={len(filtered_df)}):")
display(filtered_df.head())

# Normalization
filtered_df = filtered_df.copy()  # Avoid SettingWithCopyWarning
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Pick a group/categorical field - try 'gender', 'village', or any string field with few unique values
group_field = None
for potential in ['gender', 'sex', 'village', 'marital_status', 'education', 'group']:
    for col in df.columns:
        if potential.lower() in col.lower() and df[col].nunique() < 20:
            group_field = col
            break
    if group_field:
        break

if group_field:
    print(f"Grouping by field: {group_field}")
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print("Grouped data mean:")
    display(grouped_df.head())
else:
    print("No suitable group field found for demonstration.")

## 5. Visualization
Visualize data distributions and any notable groupings using `matplotlib`/`seaborn`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of selected numeric field
plt.figure(figsize=(8,4))
sns.histplot(df[numeric_field], kde=True, bins=20)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel("Count")
plt.show()

# Bar plot by group if group_field is available
if group_field:
    plt.figure(figsize=(8,4))
    group_means = filtered_df.groupby(group_field)[numeric_field].mean().sort_values()
    group_means.plot(kind='bar')
    plt.title(f"Mean {numeric_field} by {group_field} (filtered)")
    plt.ylabel(f"Mean {numeric_field}")
    plt.show()

## 6. Conclusion
In this notebook, we've demonstrated how to load a FAIR²-compliant dataset using `mlcroissant`, explored its structure, extracted survey records by their record set and field `@id`, performed filtering and normalization of numeric survey responses, and carried out basic visualization of the mental health indicators. This workflow forms a robust baseline for further, more advanced analysis or machine learning applications using well-documented, AI-ready data formats.